# Oracle: LightGBM quantile models + CQR calibration

Runs `organs/oracle/training/train_gbm.py` on the committed demo Parquet (modeled GloFAS discharge via Open-Meteo, ERA5 rainfall). Works on the free Colab **CPU** runtime or a laptop; no GPU is needed (about 4 minutes on an i5). Outputs go to `organs/oracle/models/` and, on Colab, are copied to Drive.

Set `COMMIT` to the commit you want to reproduce.

In [ ]:
# Pinned versions (match uv.lock)
COMMIT = "main"  # replace with a commit sha to reproduce exactly
REPO = "<public repo URL>"  # the repo has no remote configured yet; set it here
!pip -q install lightgbm==4.7.0 numpy==2.5.5 pandas==3.1.1 pyarrow==26.0.1
!pip -q install duckdb==2.0.0 pyyaml "pydantic>=2.7,<3"

In [ ]:
import os
import subprocess

if not os.path.isdir("stitch"):
    subprocess.run(["git", "clone", REPO, "stitch"], check=True)
os.chdir("stitch")
subprocess.run(["git", "checkout", COMMIT], check=True)
!pip -q install -e packages/stitch_common

In [ ]:
# --resume skips boosters already written with the same config hash (safe after a disconnect)
!python organs/oracle/training/train_gbm.py --resume

In [ ]:
import hashlib
import json
import pathlib

m = json.loads(pathlib.Path("organs/oracle/models/manifest.json").read_text())
for name, sha in m["files"].items():
    assert (
        hashlib.sha256(pathlib.Path("organs/oracle/models", name).read_bytes()).hexdigest() == sha
    ), name
print("manifest verified, root", m["root_sha256"])

In [ ]:
# Colab only: copy the bundle to Drive
try:
    from google.colab import drive

    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/stitch-oracle"
    !mkdir -p {dest} && cp -r organs/oracle/models {dest}/
except ImportError:
    print("not on Colab; models are in organs/oracle/models")